## 08 - Label validation

Checks the 13 labels of 07 in two ways. Part 1 trains the model of 06 again with other UMAP seeds and measures how many reviews keep their label. Part 2 draws a random sample of reviews for blind coding by hand, to be compared with the model labels later.

Input: `data/processed/trenord_reviews_clean.csv` (Italian reviews only) and `data/processed/trenord_reviews_labeled.csv` (from 07)
Output: stability table in this notebook, `labeling/validation_sample.xlsx` and `labeling/validation_key.csv` (not published, they contain review texts)

In [1]:
# --- IMPORTS ---
import os

import pandas as pd
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from hdbscan import HDBSCAN
from openpyxl.styles import Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.it.stop_words import STOP_WORDS as italian_stopwords
from umap import UMAP

# UMAP seeds to train. 42 is the seed of 06, the others are compared with it.
UMAP_SEEDS = [42, 0, 1, 2, 3]
# Reviews drawn from each label (and from EXCLUDE) for the blind coding sample
SAMPLE_PER_LABEL = 14
# Fixed seed for the sample and its order, so the same sample comes out on every run
SAMPLE_SEED = 42
# The two coding files. labeling/ is in .gitignore, because the files contain review texts.
SAMPLE_PATH = "../labeling/validation_sample.xlsx"
KEY_PATH = "../labeling/validation_key.csv"

W1105 15:34:56.837000 20164 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# --- LOAD DATA ---
# The cleaned reviews, with the same filter as 06: Italian reviews only
clean_df = pd.read_csv("../data/processed/trenord_reviews_clean.csv")
italian_df = clean_df[clean_df["language"] == "it"].copy()

# The label of each review, saved by 07. It also keeps the topic number given by 06.
labeled_df = pd.read_csv("../data/processed/trenord_reviews_labeled.csv")

# The two files must list the same reviews in the same order, because the topics of the new
# runs are compared with the labels position by position.
same_reviews = italian_df["review_id"].tolist() == labeled_df["review_id"].tolist()
assert same_reviews, "The clean and labeled files do not list the same reviews in the same order"

print(f"Italian reviews : {len(italian_df):,}")
print(f"Labeled reviews : {len(labeled_df):,}")
print(f"Same reviews, same order: {same_reviews}")

Italian reviews : 6,951
Labeled reviews : 6,951
Same reviews, same order: True


In [3]:
# --- PREPARE TEXTS ---
# Same text column as 06: text_clean, in natural form for the embeddings.
# BERTopic cannot handle empty strings, so they are replaced by a single space, as in 06.
texts = italian_df["text_clean"].tolist()
for i in range(len(texts)):
    text = texts[i]
    is_valid = isinstance(text, str) and len(text.strip()) > 0
    if not is_valid:
        texts[i] = " "

print(f"Texts prepared: {len(texts):,}")

Texts prepared: 6,951


In [4]:
# --- SENTENCE EMBEDDINGS (computed once) ---
# Same model as 06. The embeddings do not depend on the UMAP seed, so they are computed
# once and passed to every run. 05 showed that this gives the same topics as 06.
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
embeddings = embedding_model.encode(texts, show_progress_bar=False)

print(f"Embeddings: {embeddings.shape[0]} reviews x {embeddings.shape[1]} dimensions")

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 122.05it/s]


Embeddings: 6951 reviews x 768 dimensions


In [5]:
# --- FUNCTION TO TRAIN THE MODEL OF 06 WITH A GIVEN UMAP SEED ---
def train_topic_model(umap_random_state):
    # Every setting is the same as in 06; only the UMAP seed changes.
    # Each run gets new model objects, because BERTopic fits them in place.
    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                      random_state=umap_random_state)
    hdbscan_model = HDBSCAN(min_cluster_size=30, metric="euclidean", prediction_data=True)
    vectorizer_model = CountVectorizer(stop_words=list(italian_stopwords), min_df=5, ngram_range=(1, 2))
    representation_model = MaximalMarginalRelevance(diversity=0.3)

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        top_n_words=10,
        language="multilingual",
        calculate_probabilities=False,
        verbose=False
    )

    # Passing the embeddings skips the encoding step inside BERTopic
    topics_before, _ = topic_model.fit_transform(texts, embeddings=embeddings)

    # Outliers are reassigned to the closest topic, as in 06
    topics_after = topic_model.reduce_outliers(
        texts,
        topics_before,
        strategy="embeddings",
        threshold=0.0,
        embeddings=embeddings
    )
    return topics_before, topics_after

In [6]:
# --- TRAIN ONE MODEL PER SEED ---
# Only the topic of each review is kept: the stability check needs nothing else.
topics_by_seed = {}
for seed in UMAP_SEEDS:
    topics_before, topics_after = train_topic_model(seed)
    topics_by_seed[seed] = list(topics_after)

    n_topics = len(set(topics_after))
    n_outliers_before = list(topics_before).count(-1)
    print(f"seed {seed:2d}: {n_topics} topics, {n_outliers_before:,} outliers before reduction")

seed 42: 30 topics, 1,601 outliers before reduction
seed  0: 33 topics, 2,023 outliers before reduction
seed  1: 27 topics, 1,264 outliers before reduction
seed  2: 31 topics, 1,564 outliers before reduction
seed  3: 27 topics, 984 outliers before reduction


In [7]:
# --- SEED 42 GIVES THE TOPICS OF 06 ---
# Seed 42 has exactly the settings of 06, so every review must get the topic saved in
# trenord_reviews_labeled.csv. If it does, the other seeds differ from 06 only by the seed.
labeled_topics = labeled_df["topic"].tolist()

n_different = 0
for new_topic, saved_topic in zip(topics_by_seed[42], labeled_topics):
    if new_topic != saved_topic:
        n_different += 1

if n_different == 0:
    result = "PASS"
else:
    result = "FAIL"
print(f"{result}  seed 42: {n_different} of {len(labeled_topics):,} reviews have a different topic from trenord_reviews_labeled.csv")

PASS  seed 42: 0 of 6,951 reviews have a different topic from trenord_reviews_labeled.csv


### How the stability of the labels is measured

A new seed gives new topics with new numbers, so its topics cannot be compared with the old ones by number. Instead, each new topic gets the current label (one of the 13, or EXCLUDE) that most of its reviews have now. Then each review is checked: does its new topic carry the same label it has now?

A made-up example. With seed 0, topic 4 has 100 reviews: 70 are now Ticket Purchase Failures, 20 Subscription Purchase Failures and 10 EXCLUDE. Topic 4 gets Ticket Purchase Failures. The 70 keep their label; the other 30 change label, even if their text did not change at all.

If a label is the majority in no new topic, all its reviews change label in that seed, and its share is 0%. The overall row counts every review, so large labels weigh more in it.

The measure is generous: each new topic takes the label that fits most of its reviews, so the shares are as high as this mapping allows. It shows whether the themes come back with another seed. It does not show whether a person reading the new topics would give them the same names.

In [8]:
# --- GIVE EACH TOPIC OF THE OTHER SEEDS A CURRENT LABEL ---
current_labels = labeled_df["topic_label"].tolist()

mapped_labels_by_seed = {}
for seed in UMAP_SEEDS:
    if seed == 42:
        continue

    seed_df = pd.DataFrame({"seed_topic": topics_by_seed[seed], "current_label": current_labels})

    label_of_seed_topic = {}
    n_ties = 0
    for seed_topic in sorted(seed_df["seed_topic"].unique()):
        topic_reviews = seed_df[seed_df["seed_topic"] == seed_topic]
        # value_counts() sorts from the most frequent label, so the first one is the majority
        label_counts = topic_reviews["current_label"].value_counts()
        # A tie means two labels have the same count; the first one is then taken
        if len(label_counts) > 1 and label_counts.iloc[0] == label_counts.iloc[1]:
            n_ties += 1
        label_of_seed_topic[seed_topic] = label_counts.index[0]

    mapped_labels = seed_df["seed_topic"].map(label_of_seed_topic).tolist()
    mapped_labels_by_seed[seed] = mapped_labels

    n_labels_reached = len(set(label_of_seed_topic.values()))
    print(f"seed {seed}: {len(label_of_seed_topic)} topics, {n_labels_reached} of 14 labels are the majority in at least one topic, {n_ties} ties")

seed 0: 33 topics, 14 of 14 labels are the majority in at least one topic, 0 ties
seed 1: 27 topics, 14 of 14 labels are the majority in at least one topic, 0 ties
seed 2: 31 topics, 14 of 14 labels are the majority in at least one topic, 0 ties
seed 3: 27 topics, 12 of 14 labels are the majority in at least one topic, 0 ties


In [9]:
# --- SHARE OF REVIEWS THAT KEEP THEIR LABEL ---
# One row per current label, largest first, and one column per seed.
label_order = labeled_df["topic_label"].value_counts().index.tolist()

rows = []
for label in label_order:
    row = {"label": label, "reviews": current_labels.count(label)}
    for seed, mapped_labels in mapped_labels_by_seed.items():
        n_same = 0
        for current, mapped in zip(current_labels, mapped_labels):
            if current == label and mapped == label:
                n_same += 1
        row[f"seed {seed} (%)"] = round(n_same / row["reviews"] * 100, 1)
    rows.append(row)

# The overall row: the share of all reviews whose label stays the same
overall_row = {"label": "Overall", "reviews": len(current_labels)}
for seed, mapped_labels in mapped_labels_by_seed.items():
    n_same = 0
    for current, mapped in zip(current_labels, mapped_labels):
        if current == mapped:
            n_same += 1
    overall_row[f"seed {seed} (%)"] = round(n_same / len(current_labels) * 100, 1)
rows.append(overall_row)

stability_table = pd.DataFrame(rows).set_index("label")
stability_table

,reviews,seed 0 (%),seed 1 (%),seed 2 (%),seed 3 (%)
label,,,,,
Unreliable Train Status,1501,98.6,97.1,95.5,93.5
Ticket Purchase Failures,1143,93.8,97.8,98.4,99.4
EXCLUDE,953,96.2,81.3,86.6,82.9
App Crashes & Freezes,655,91.3,92.4,88.2,83.7
Subscription Purchase Failures,641,95.6,98.3,94.4,0.0
Poor Usability,499,86.6,97.4,98.2,96.6
Ease of Use,463,96.3,96.5,98.3,95.0
Login & Account Issues,289,94.1,95.2,91.0,0.0
Easy Purchase & Payment,277,94.2,93.1,89.9,82.3


In [10]:
# --- WHERE THE REVIEWS GO WHEN A LABEL IS LOST ---
# For the labels that keep fewer than half of their reviews in a seed: the labels their
# reviews get instead, with their shares, the largest three only.
for seed, mapped_labels in mapped_labels_by_seed.items():
    for label in label_order:
        share_kept = stability_table.loc[label, f"seed {seed} (%)"]
        if share_kept >= 50:
            continue

        new_labels = []
        for current, mapped in zip(current_labels, mapped_labels):
            if current == label:
                new_labels.append(mapped)
        new_label_shares = pd.Series(new_labels).value_counts(normalize=True) * 100

        parts = []
        for new_label, share in new_label_shares.head(3).items():
            parts.append(f"{new_label} {share:.1f}%")
        print(f"seed {seed}, {label} ({share_kept}% kept): " + ", ".join(parts))

seed 3, Subscription Purchase Failures (0.0% kept): Ticket Purchase Failures 98.8%, Unreliable Train Status 0.8%, App Crashes & Freezes 0.3%
seed 3, Login & Account Issues (0.0% kept): Ticket Purchase Failures 96.9%, Poor Usability 1.4%, Unreliable Train Status 0.7%


### Reading the stability table

- With seeds 0, 1 and 2, 93.6% to 94.6% of the reviews keep their label, and every label keeps at least 81.3% of its reviews.
- Seed 3 is different: 79.1% overall. Subscription Purchase Failures and Login & Account Issues are the majority in no topic, so they keep 0%. 98.8% of the subscription reviews and 96.9% of the login reviews end up in topics where Ticket Purchase Failures is the majority. With this seed, problems with buying, paying and logging in are not separated. Seed 3 is also the seed with the lowest ARI against seed 42 in 05 (0.515).
- The two largest labels are the most stable: Unreliable Train Status keeps 93.5% to 98.6% of its reviews, Ticket Purchase Failures 93.8% to 99.4%.
- EXCLUDE (81.3% to 96.2%) and Rail Service Complaints (79.3% to 88.5%) move the most among the labels that never disappear. Short verdicts and complaints about the company share many words with the other topics.

So most themes come back with every seed, but the split between ticket, subscription and login problems depends on the seed: in one seed out of four it is gone.

## Part 2 - The blind coding sample

The labels of 07 were given by reading the keywords and some reviews of each topic. To check them, a sample of reviews is coded by hand without seeing the model label, and the two are compared later.

The sample is stratified: 14 reviews from each of the 13 labels and 14 from the EXCLUDE reviews, 196 in total. A simple random sample would take mostly Unreliable Train Status and Ticket Purchase Failures, and only one or two reviews from the small labels such as Missing Stations, too few to say anything about them. With the same number from each label, every label can be checked.

The price is that the sample does not have the proportions of all reviews. The share of matches in the sample over-weights the small labels, so an accuracy for all reviews will need each label weighted by its real share of reviews.

In [11]:
# --- CODEBOOK ---
# One line per label, taken from the comments of the label map in 07. When a label has more
# than one topic, the comments are joined. The coder reads it before coding.
CODEBOOK = {
    "Unreliable Train Status": ("App Issue", "Delays and cancellations shown late, wrong or not at all"),
    "Ticket Purchase Failures": ("App Issue", "Tickets that cannot be bought, errors at checkout"),
    "Subscription Purchase Failures": ("App Issue", "Subscription renewal and payment errors"),
    "Poor Usability": ("App Issue", "The app is slow or unclear to use"),
    "App Crashes & Freezes": ("App Issue", "The app stops working, often after an update"),
    "Login & Account Issues": ("App Issue", "Login, account and registration problems"),
    "Station & Favorites Bugs": ("App Issue", "Favourite stations lost, station search not working"),
    "QR Code Failures": ("App Issue", "QR code of the ticket missing or not read at the metro gates"),
    "Missing Stations": ("App Issue", "Stations removed from the app or missing from its list"),
    "Ease of Use": ("Praise", "The app is convenient and easy to use"),
    "Easy Purchase & Payment": ("Praise", "Buying tickets or paying for subscriptions in the app is easy and fast"),
    "Version Comparisons": ("App Updates", "Comparisons with earlier versions of the app"),
    "Rail Service Complaints": ("Rail Service", "About the trains and the company, not the app"),
    "EXCLUDE": ("", "Short generic verdict with no specific content"),
    "None of these": ("", "The review fits no label"),
}

# The codebook must hold exactly the labels of 07, plus EXCLUDE and "None of these"
labels_in_file = set(labeled_df["topic_label"])
labels_in_codebook = set(CODEBOOK) - {"None of these"}
assert labels_in_codebook == labels_in_file, "The codebook and trenord_reviews_labeled.csv have different labels"

print(f"Labels in the codebook: {len(CODEBOOK)} (13 labels, EXCLUDE and None of these)")

Labels in the codebook: 15 (13 labels, EXCLUDE and None of these)


In [12]:
# --- DRAW THE STRATIFIED SAMPLE ---
# The same number of reviews from each label, drawn at random with a fixed seed.
# The labels follow the order of the codebook, so the stacked table, and the shuffle of the next cell,
# come out the same on every run.
sample_parts = []
for label in CODEBOOK:
    if label == "None of these":
        continue
    label_reviews = labeled_df[labeled_df["topic_label"] == label]
    sample_parts.append(label_reviews.sample(n=SAMPLE_PER_LABEL, random_state=SAMPLE_SEED))

sample_df = pd.concat(sample_parts)

print(f"Reviews in the sample : {len(sample_df)}")
print(f"Distinct reviews      : {sample_df['review_id'].nunique()}")
print()
print("Reviews per label:")
print(sample_df["topic_label"].value_counts().to_string())

Reviews in the sample : 196
Distinct reviews      : 196

Reviews per label:
topic_label
Unreliable Train Status           14
Ticket Purchase Failures          14
Subscription Purchase Failures    14
Poor Usability                    14
App Crashes & Freezes             14
Login & Account Issues            14
Station & Favorites Bugs          14
QR Code Failures                  14
Missing Stations                  14
Ease of Use                       14
Easy Purchase & Payment           14
Version Comparisons               14
Rail Service Complaints           14
EXCLUDE                           14


In [13]:
# --- SHUFFLE THE ORDER ---
# Without shuffling, the 14 reviews of each label would come one after the other, and the
# coder could guess the label from the neighbours. sample(frac=1) returns all rows in random order.
sample_df = sample_df.sample(frac=1, random_state=SAMPLE_SEED).reset_index(drop=True)

# The id is given after shuffling, so it says nothing about the label
sample_df["sample_id"] = range(1, len(sample_df) + 1)

# Longest run of reviews with the same label in a row, to check that labels are not in blocks
longest_run = 1
current_run = 1
sample_labels = sample_df["topic_label"].tolist()
for i in range(1, len(sample_labels)):
    if sample_labels[i] == sample_labels[i - 1]:
        current_run += 1
    else:
        current_run = 1
    if current_run > longest_run:
        longest_run = current_run

# Reviews with no text cannot be coded, so they are counted
n_empty = 0
for text in sample_df["text"]:
    if not isinstance(text, str) or len(text.strip()) == 0:
        n_empty += 1

print(f"Longest run of the same label in a row: {longest_run}")
print(f"Reviews with an empty text            : {n_empty}")

Longest run of the same label in a row: 2
Reviews with an empty text            : 0


In [14]:
# --- WRITE THE CODING FILES (only the first time) ---
# If the Excel file is already there, nothing is written, so a rerun never overwrites the coding.
if os.path.exists(SAMPLE_PATH):
    print("labeling/validation_sample.xlsx already exists: no file was written, the coding is kept")
else:
    os.makedirs("../labeling", exist_ok=True)

    # Sheet "coding": the review and an empty column for my label. The model label is not here.
    coding_sheet = sample_df[["sample_id", "text"]].copy()
    coding_sheet["my_label"] = None

    # Sheet "codebook": one row per label with its category and definition
    codebook_rows = []
    for label, entry in CODEBOOK.items():
        codebook_rows.append({"label": label, "category": entry[0], "definition": entry[1]})
    codebook_sheet = pd.DataFrame(codebook_rows)

    with pd.ExcelWriter(SAMPLE_PATH, engine="openpyxl") as writer:
        coding_sheet.to_excel(writer, sheet_name="coding", index=False)
        codebook_sheet.to_excel(writer, sheet_name="codebook", index=False)

        # The dropdown reads its values from the label column of the codebook sheet.
        # A list written inside the rule would be longer than the 255 characters Excel allows.
        worksheet = writer.sheets["coding"]
        last_codebook_row = len(codebook_sheet) + 1
        dropdown = DataValidation(type="list", formula1=f"codebook!$A$2:$A${last_codebook_row}",
                                  allow_blank=True, showErrorMessage=True)
        worksheet.add_data_validation(dropdown)
        dropdown.add(f"C2:C{len(coding_sheet) + 1}")

        # Wide, wrapped text column, so long reviews can be read without resizing
        worksheet.column_dimensions["B"].width = 100
        worksheet.column_dimensions["C"].width = 32
        for row in worksheet.iter_rows(min_row=2, min_col=2, max_col=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

        codebook_worksheet = writer.sheets["codebook"]
        codebook_worksheet.column_dimensions["A"].width = 32
        codebook_worksheet.column_dimensions["B"].width = 14
        codebook_worksheet.column_dimensions["C"].width = 70

    # The hidden key: the model label of each sample_id. It stays closed until the coding is done.
    key_df = sample_df[["sample_id", "topic_label"]].rename(columns={"topic_label": "model_label"})
    key_df.to_csv(KEY_PATH, index=False, encoding="utf-8-sig")

    print("Created labeling/validation_sample.xlsx and labeling/validation_key.csv")

Created labeling/validation_sample.xlsx and labeling/validation_key.csv


In [15]:
# --- CHECK THE CODING FILES ---
# The files are read back from disk. Only counts are printed, so the key stays hidden.
coding_back = pd.read_excel(SAMPLE_PATH, sheet_name="coding")
codebook_back = pd.read_excel(SAMPLE_PATH, sheet_name="codebook")
key_back = pd.read_csv(KEY_PATH)

# Same reviews in the same order as the sample drawn above. If the data has changed since
# the files were made, this is False and the files describe an older sample.
texts_in_file = coding_back["text"].fillna("").tolist()
texts_in_sample = sample_df["text"].fillna("").tolist()
same_texts = texts_in_file == texts_in_sample

# Same ids in the coding sheet and in the key
same_ids = coding_back["sample_id"].tolist() == key_back["sample_id"].tolist()

# Each label must have the same number of reviews in the key
key_counts = key_back["model_label"].value_counts()
equal_counts = (key_counts == SAMPLE_PER_LABEL).all() and len(key_counts) == 14

n_coded = coding_back["my_label"].notna().sum()

print(f"Rows in the coding sheet  : {len(coding_back)}")
print(f"Rows in the codebook sheet: {len(codebook_back)}")
print(f"Rows in the key           : {len(key_back)}")
print(f"Same texts as the sample  : {same_texts}")
print(f"Same ids in sheet and key : {same_ids}")
print(f"{SAMPLE_PER_LABEL} reviews for each of the 14 labels in the key: {equal_counts}")
print(f"Reviews coded so far      : {n_coded}")

Rows in the coding sheet  : 196
Rows in the codebook sheet: 15
Rows in the key           : 196
Same texts as the sample  : True
Same ids in sheet and key : True
14 reviews for each of the 14 labels in the key: True
Reviews coded so far      : 0


### Next step

The sample is coded by hand in `labeling/validation_sample.xlsx`: one label per review from the dropdown of `my_label`, with the codebook sheet at hand, and without opening `labeling/validation_key.csv`. Once all 196 reviews are coded, my labels will be compared with the key, label by label, and the accuracy for all reviews will weight each label by its share of reviews.